# Streaming  流媒体
### Basic usage  基本用法
stream （同步）和 astream （异步）方法，用于以迭代器的形式生成流式输出。可以传递一个或多个 stream modes(流模式)来控制接收的数据。

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer


class State(TypedDict):
    topic: str
    joke: str


def generate_joke(state: State):
    writer = get_stream_writer()#get_stream_writer()获取一个可以向外部发送 streaming event 的 writer。
    writer({"Thinking": "thinking of a joke..."})#通过 writer 发送一个自定义事件，通知外部我们正在思考一个笑话。
    return {"joke": f"Why did the {state['topic']} go to school? To get a sundae education!"}

graph = (
    StateGraph(State)
    .add_node(generate_joke)
    .add_edge(START, "generate_joke")
    .add_edge("generate_joke", END)
    .compile()
)

inputs = {"topic": "ice cream"}

async for chunk in graph.astream(
    {"topic": "ice cream"},
    stream_mode=["updates", "custom"],
    version="v2",
):
    if chunk["type"] == "updates":#state更新输出
        print(chunk)
        for node_name, state in chunk["data"].items():
            print(f"Node {node_name} updated: {state}")

    elif chunk["type"] == "custom":#自定义输出
        print(f"Thinking: {chunk['data']['Thinking']}")

将 version="v2" 传递给 stream() 或 astream() 可获得统一的输出格式。每个数据块都是一个结构一致的 StreamPart 字典——无论流模式、模式数量或子图设置如何：
```json
{
    "type": "values" | "updates" | "messages" | "custom" | "checkpoints" | "tasks" | "debug",
    "ns": (),           # namespace tuple, populated for subgraph events
    "data": ...,        # the actual payload (type varies by stream mode)
}
```

In [ ]:
for chunk in graph.stream(inputs, stream_mode="custom", version="v2"):
    print(chunk["type"])  # "updates"
    print(chunk["ns"])    # ()
    print(chunk["data"])  # {"node_name": {"key": "value"}}

In [ ]:
for chunk in graph.stream(inputs, stream_mode="updates"):#v1
    print(chunk)  # {"node_name": {"key": "value"}}

v2 格式还支持类型缩小，这意味着可以按 chunk["type"] 筛选数据块，从而获得正确的有效负载类型。每个分支都会将 part["data"] 缩小到该模式下的特定类型：

In [ ]:
for part in graph.stream(
    {"topic": "ice cream"},
    stream_mode=["values", "updates", "messages", "custom"],
    version="v2",
):
    if part["type"] == "values":
        # ValuesStreamPart — full state snapshot after each step
        print(f"State: topic={part['data']}")
    elif part["type"] == "updates":
        # UpdatesStreamPart — only the changed keys from each node
        for node_name, state in part["data"].items():
            print(f"Node `{node_name}` updated: {state}")
    elif part["type"] == "messages":
        # MessagesStreamPart — (message_chunk, metadata) from LLM calls
        msg, metadata = part["data"]
        print(msg.content, end="", flush=True)
        print("testing")
    elif part["type"] == "custom":
        # CustomStreamPart — arbitrary data from get_stream_writer()
        print(f"Progress: {part['data']['Thinking']}")

# LangGraph Stream Modes（流式模式）

| Mode | 类型 | 描述 |
|------|------|------|
| **values** | `ValuesStreamPart` | 每一步完成后的**完整状态（full state）** |
| **updates** | `UpdatesStreamPart` | 每一步完成后的**状态增量更新**，同一步内多次更新会分别流式发送 |
| **messages** | `MessagesStreamPart` | 来自 LLM 调用的 `(token, metadata)` 二元组流 |
| **custom** | `CustomStreamPart` | 通过 `get_stream_writer()` 在节点内主动发送的自定义数据 |
| **checkpoints** | `CheckpointStreamPart` | 检查点事件（格式与 `get_state()` 一致），**需要 checkpointer** |
| **tasks** | `TasksStreamPart` | 任务开始/结束事件，包含结果与错误，**需要 checkpointer** |
| **debug** | `DebugStreamPart` | 所有可用信息（包含 checkpoints + tasks + 额外 metadata） |

### Graph state  图状态
- `updates` 会将图的每一步之后的状态更新流式传输。
- `values` 流式传输图的每一步之后的状态的完整值 。

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
  topic: str
  joke: str


def refine_topic(state: State):
    return {"topic": state["topic"] + " and cats"}


def generate_joke(state: State):
    return {"joke": f"This is a joke about {state['topic']}"}

graph = (
  StateGraph(State)
  .add_node(refine_topic)
  .add_node(generate_joke)
  .add_edge(START, "refine_topic")
  .add_edge("refine_topic", "generate_joke")
  .add_edge("generate_joke", END)
  .compile()
)

In [ ]:
# updates 模式下，外部会收到每个节点的更新状态
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        for node_name, state in chunk["data"].items():
            print(f"Node `{node_name}` updated: {state}")

In [ ]:
# values 模式下，外部会收到每个节点的完整状态
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="values",
    version="v2",
):
    if chunk["type"] == "values":
        print(chunk['data'])

### LLM tokens  LLM 代币
`messages` 模式的流式输出是一个元组 (message_chunk, metadata) ，其中：
- `message_chunk` ：来自 LLM 的标记或消息段。
- `metadata` ：包含有关图节点和 LLM 调用的详细信息的字典。

In [ ]:
from dataclasses import dataclass

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GLM_API_KEY")
base_url = os.getenv("GLM_BASE_URL")

model = ChatOpenAI(
    model="glm-4.5-air",
    api_key=api_key,
    base_url=base_url,
    temperature=0,
    streaming=True,
    tags=["joke"]
)

@dataclass
class MyState:
    topic: str
    joke: str = ""

def call_model(state: MyState):
    """Call the LLM to generate a joke about a topic"""
    # Note that message events are emitted even when the LLM is run using .invoke rather than .stream
    model_response = model.invoke(
        [
            {"role": "user", "content": f"Generate a joke about {state.topic}"}
        ]
    )
    return {"joke": model_response.content}

graph = (
    StateGraph(MyState)
    .add_node(call_model)
    .add_edge(START, "call_model")
    .compile()
)

# The "messages" stream mode streams LLM tokens with metadata
# Use version="v2" for a unified StreamPart format
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="messages",
    version="v2",
):
    if chunk["type"] == "messages":
        message_chunk, metadata = chunk["data"]
        if message_chunk.content:
            print(message_chunk.content, end="|", flush=True)


#### Filter by LLM invocation  按 LLM 调用筛选
可以将 tags 与 LLM 调用关联起来，以便按 LLM 调用筛选流式令牌。

In [ ]:
# The stream_mode is set to "messages" to stream LLM tokens
# The metadata contains information about the LLM invocation, including the tags
async for chunk in graph.astream(
    {"topic": "cats"},
    stream_mode="messages",
    version="v2",
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        # Filter the streamed tokens by the tags field in the metadata to only include
        # the tokens from the LLM invocation with the "joke" tag
        if metadata["tags"] == ["joke"]:
            print(msg.content, end="|", flush=True)


#### Filter by node  按节点筛选
要仅从特定节点流式传输令牌，请使用 stream_mode="messages" 并按流式元数据中的 langgraph_node 字段过滤输出：

In [ ]:
# The "messages" stream mode streams LLM tokens with metadata
# Use version="v2" for a unified StreamPart format
for chunk in graph.stream(
    inputs,
    stream_mode="messages",
    version="v2",
):
    if chunk["type"] == "messages":
        msg, metadata = chunk["data"]
        # Filter the streamed tokens by the langgraph_node field in the metadata
        # to only include the tokens from the specified node
        if msg.content and metadata["langgraph_node"] == "call_model":
            print(msg.content, end="|", flush=True)

### Custom data  自定义数据
从 LangGraph 节点或工具内部发送自定义用户定义数据 
1. 使用 get_stream_writer 访问流写入器并发出自定义数据。
2. 调用 .stream() 或 .astream() 时，设置 stream_mode="custom" 以在流中获取自定义数据。您可以组合多个模式（例如， ["updates", "custom"] ），但至少其中一个必须是 "custom" 。

In [ ]:
from typing import TypedDict
from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START

class State(TypedDict):
    query: str
    answer: str

def node(state: State):
    # Get the stream writer to send custom data
    writer = get_stream_writer()
    # Emit a custom key-value pair (e.g., progress update)
    writer({"custom_key": "Generating custom data inside node"})
    return {"answer": "some data"}

graph = (
    StateGraph(State)
    .add_node(node)
    .add_edge(START, "node")
    .compile()
)

inputs = {"query": "example"}

# Set stream_mode="custom" to receive the custom data in the stream
for chunk in graph.stream(inputs, stream_mode="custom", version="v2"):
    if chunk["type"] == "custom":
        print(f"Custom event: {chunk['data']['custom_key']}")

> 在 Python 3.11 及更高版本中， asyncio 任务不支持 context 参数。这限制了 LangGraph 自动传播上下文的能力，并从两个关键方面影响 LangGraph 的流式传输机制：
> 1. 您必须显式地将 RunnableConfig 传递给异步 LLM 调用（例如 ainvoke() ），因为回调不会自动传播。
> 2. 不能在异步节点或工具中使用 get_stream_writer ，必须直接传递 writer 参数。

In [ ]:
from typing import TypedDict
from langgraph.graph import START, StateGraph
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

class State(TypedDict):
    topic: str
    joke: str

# Accept config as an argument in the async node function
async def call_model(state, config):#扩展示例：带手动配置的异步 LLM 调用
    topic = state["topic"]
    print("Generating joke...")
    # Pass config to model.ainvoke() to ensure proper context propagation
    joke_response = await model.ainvoke(
        [{"role": "user", "content": f"Write a joke about {topic}"}],
        config,
    )
    return {"joke": joke_response.content}

graph = (
    StateGraph(State)
    .add_node(call_model)
    .add_edge(START, "call_model")
    .compile()
)

# Set stream_mode="messages" to stream LLM tokens
async for chunk in graph.astream(
    {"topic": "ice cream"},
    stream_mode="messages",
    version="v2",
):
    if chunk["type"] == "messages":
        message_chunk, metadata = chunk["data"]
        if message_chunk.content:
            print(message_chunk.content, end="|", flush=True)

In [ ]:
from typing import TypedDict
from langgraph.types import StreamWriter

class State(TypedDict):
      topic: str
      joke: str

# Add writer as an argument in the function signature of the async node or tool
# LangGraph will automatically pass the stream writer to the function
async def generate_joke(state: State, writer: StreamWriter):#扩展示例：使用流写入器实现异步自定义流式传输
      writer({"custom_key": "Streaming custom data while generating a joke"})
      return {"joke": f"This is a joke about {state['topic']}"}

graph = (
      StateGraph(State)
      .add_node(generate_joke)
      .add_edge(START, "generate_joke")
      .compile()
)

# Set stream_mode="custom" to receive the custom data in the stream  #
async for chunk in graph.astream(
      {"topic": "ice cream"},
      stream_mode="custom",
      version="v2",
):
      if chunk["type"] == "custom":
          print(chunk["data"])

### Subgraph outputs  子图输出
要将子图的输出包含在流式输出中，可以在父图的 .stream() 方法中设置 subgraphs=True 。这将同时流式传输父图及其所有子图的输出。
输出将以元组 (namespace, data) 的形式流式传输，其中 namespace 是一个元组，包含调用子图的节点的路径，例如 ("parent_node:<task_id>", "child_node:<task_id>") 。

In [1]:
from langgraph.graph import START, StateGraph
from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver

# Define subgraph
class SubgraphState(TypedDict):
    foo: str  # note that this key is shared with the parent graph state
    bar: str

def subgraph_node_1(state: SubgraphState):
    return {"bar": "bar"}

def subgraph_node_2(state: SubgraphState):
    return {"foo": state["foo"] + state["bar"]}

subgraph_builder = StateGraph(SubgraphState)
subgraph_builder.add_node(subgraph_node_1)
subgraph_builder.add_node(subgraph_node_2)
subgraph_builder.add_edge(START, "subgraph_node_1")
subgraph_builder.add_edge("subgraph_node_1", "subgraph_node_2")
subgraph = subgraph_builder.compile(checkpointer=MemorySaver())

# Define parent graph
class ParentState(TypedDict):
    foo: str

def node_1(state: ParentState):
    return {"foo": "hi! " + state["foo"]}

builder = StateGraph(ParentState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", subgraph)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
graph = builder.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "1"}}  # example config to demonstrate context propagation to subgraph

for chunk in graph.stream(
    {"foo": "foo"},
    stream_mode="updates",
    # Set subgraphs=True to stream outputs from subgraphs
    subgraphs=True,
    version="v2",
    config=config
):
    if chunk["type"] == "updates":
        if chunk["ns"]:
            print(f"Subgraph {chunk['ns']}: {chunk['data']}")
        else:
            print(f"Root: {chunk['data']}")

Root: {'node_1': {'foo': 'hi! foo'}}
Subgraph ('node_2:a6cf602c-8649-40c2-ebe5-88cbb7f9e490',): {'subgraph_node_1': {'bar': 'bar'}}
Subgraph ('node_2:a6cf602c-8649-40c2-ebe5-88cbb7f9e490',): {'subgraph_node_2': {'foo': 'hi! foobar'}}
Root: {'node_2': {'foo': 'hi! foobar'}}


### 

### Checkpoints  检查点
使用 checkpoints 流模式可在图执行过程中接收检查点事件。每个检查点事件的格式与 get_state() 的输出格式相同。需要检查点器 。

In [ ]:
config = {"configurable": {"thread_id": "1"}}

for chunk in graph.stream(
    {"foo": "foo"},
    config=config,
    stream_mode="checkpoints",
    version="v2",
):
    if chunk["type"] == "checkpoints":
        print(chunk["data"])

### Tasks  任务
使用 tasks 流模式可以在图执行过程中接收任务的启动和完成事件。任务事件包含有关哪个节点正在运行、其结果以及任何错误的信息。需要检查点 。

In [ ]:
config = {"configurable": {"thread_id": "1"}}

for chunk in graph.stream(
    {"topic": "ice cream"},
    config=config,
    stream_mode="tasks",
    version="v2",
):
    if chunk["type"] == "tasks":
        print(chunk["data"])

### Debug  调试
使用 debug 流模式可以在图的执行过程中尽可能多地传输信息。传输的输出包括节点名称和完整状态。


In [ ]:
for chunk in graph.stream(
    {"foo": "foo"},
    stream_mode="debug",
    config=config,
    version="v2",
):
    if chunk["type"] == "debug":
        print(chunk["data"])

### Multiple modes at once  同时存在多种模式
你可以将一个列表作为 stream_mode 参数传递，以便同时传输多种模式。
使用 version="v2" 时，每个数据块都是一个 StreamPart 字典。使用 `chunk["type"]` 来区分不同的模式：

In [ ]:
for chunk in graph.stream({"foo": "foo"}, stream_mode=["updates", "custom"], version="v2",config=config):
    if chunk["type"] == "updates":
        for node_name, state in chunk["data"].items():
            print(f"Node `{node_name}` updated: {state}")
    elif chunk["type"] == "custom":
        print(f"Custom event: {chunk['data']}")

## Advanced  先进的
### Use with any LLM  可与任何法学硕士课程配合使用
可以使用 stream_mode="custom" 从任何 LLM API 流式传输数据——即使该 API 没有实现 LangChain 聊天模型接口。

In [ ]:
from langgraph.config import get_stream_writer

def call_arbitrary_model(state):
    """Example node that calls an arbitrary model and streams the output"""
    # Get the stream writer to send custom data
    writer = get_stream_writer()
    # Assume you have a streaming client that yields chunks
    # Generate LLM tokens using your custom streaming client
    for chunk in your_custom_streaming_client(state["topic"]): # pyright: ignore[reportUndefinedVariable]
        # Use the writer to send custom data to the stream
        writer({"custom_llm_chunk": chunk})
    return {"result": "completed"}

graph = (
    StateGraph(State)
    .add_node(call_arbitrary_model)
    # Add other nodes and edges as needed
    .compile()
)
# Set stream_mode="custom" to receive the custom data in the stream
for chunk in graph.stream(
    {"topic": "cats"},
    stream_mode="custom",
    version="v2",
):
    if chunk["type"] == "custom":
        # The chunk data will contain the custom data streamed from the llm
        print(chunk["data"])

| 场景 | v1（默认） | v2（version="v2"） |
|------|------------|--------------------|
| 单一流模式 | 原始数据（dict） | StreamPart 字典，包含 type、ns、data |
| 多流模式 | (mode, data) 元组 | 同样是 StreamPart 字典，通过 chunk["type"] 过滤 |
| 子图流式输出 | (namespace, data) 元组 | 同样是 StreamPart 字典，通过 chunk["ns"] 判断 |
| 多模式 + 子图 | (namespace, mode, data) 三元组 | 同样是 StreamPart 字典 |
| invoke() 返回类型 | 普通 dict（state） | GraphOutput，包含 `.value` 和 `.interrupts` |
| Interrupt 位置（stream） | state 字典中的 `__interrupt__` 键 | values 流分片中的 `interrupts` 字段 |
| Interrupt 位置（invoke） | result 字典中的 `__interrupt__` 键 | GraphOutput 的 `.interrupts` 属性 |
| Pydantic / dataclass 输出 | 返回普通 dict | 自动转换为对应的 model / dataclass 实例 |

In [3]:
from langgraph.types import GraphOutput

result = graph.invoke({"foo": "foo"}, version="v2",config=config)

assert isinstance(result, GraphOutput)
result.value       # your output — dict, Pydantic model, or dataclass
result.interrupts  # tuple[Interrupt, ...], empty if none occurred

()